# 1. Imports

In [2]:
import pandas as pd

# 2. Loading Data

In [3]:
df = pd.read_csv("../data/hdb_data.csv")
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235808 entries, 0 to 235807
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                235808 non-null  str    
 1   town                 235808 non-null  str    
 2   flat_type            235808 non-null  str    
 3   block                235808 non-null  str    
 4   street_name          235808 non-null  str    
 5   storey_range         235808 non-null  str    
 6   floor_area_sqm       235808 non-null  float64
 7   flat_model           235808 non-null  str    
 8   lease_commence_date  235808 non-null  int64  
 9   remaining_lease      235808 non-null  str    
 10  resale_price         235808 non-null  float64
dtypes: float64(2), int64(1), str(8)
memory usage: 19.8 MB


In [5]:
df.describe()

,floor_area_sqm,lease_commence_date,resale_price
count,235808.000000,235808.000000,2.358080e+05
mean,96.686015,1996.573475,5.323648e+05
std,24.016888,14.370177,1.909245e+05
min,31.000000,1966.000000,1.400000e+05
25%,81.000000,1985.000000,3.900000e+05
50%,93.000000,1997.000000,5.000000e+05
75%,112.000000,2012.000000,6.400000e+05
max,366.700000,2022.000000,1.728000e+06


# 3. Cleaning Data

In [6]:
#Checking for missing values
print(df.isnull().sum()) # There are no missing values in the dataset

#Checking for duplicates
print(df.duplicated().sum()) # There are 316 duplicates in the dataset


# Checking Duplicates
duplicates = df[df.duplicated(keep=False)] #manually inspecting duplicates
print(duplicates)

print(df.duplicated().sum() / len(df)) #checking % of df that are duplicates (0.13%)

# Dropping duplicates
clean_df = df.drop_duplicates() # dropping duplicates in a new clean dataframe
print(clean_df.duplicated().sum())



month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64
316
          month          town flat_type block       street_name storey_range  \
224     2017-01   BUKIT MERAH    4 ROOM   106    HENDERSON CRES     07 TO 09   
243     2017-01   BUKIT MERAH    4 ROOM   106    HENDERSON CRES     07 TO 09   
304     2017-01  CENTRAL AREA    3 ROOM   271          QUEEN ST     16 TO 18   
305     2017-01  CENTRAL AREA    3 ROOM   271          QUEEN ST     16 TO 18   
505     2017-01   JURONG EAST    4 ROOM   265       TOH GUAN RD     04 TO 06   
...         ...           ...       ...   ...               ...          ...   
231833  2026-03      SENGKANG    5 ROOM  224C  COMPASSVALE WALK     07 TO 09   
233624  2026-03     TOA PAYOH    4 ROOM  103B    BIDADARI PK DR     

# 3.1 Transforming Columns 

In [7]:
#transforming the month column
clean_df['month'] = pd.to_datetime(clean_df['month'], format='%Y-%m') # changing month column to date time 
clean_df['date'] = clean_df['month'] # creating date column with original value
clean_df['year'] = clean_df['month'].dt.year #creating year column with year value 
clean_df['month'] = clean_df['month'].dt.month # creating month column with month value 

#transforming storey range column 
clean_df['storey_range'] = clean_df['storey_range'].str.split(' TO ', expand=True).astype(int).mean(axis=1) # splits storey range in half, turns values to integers and finds the mean
print(clean_df['storey_range'])


#transforming the remaining lease column
clean_df['remaining_lease'] = clean_df['remaining_lease'].str.replace(' months', '') # replacing months with blank space
clean_df['remaining_lease'] = clean_df['remaining_lease'].str.replace(' month', '') # replacing month with blank space to catch "1 month"
clean_df['remaining_lease'] = (clean_df['remaining_lease'].str.replace('years', '-')) #replacing years with - to split with later 
clean_df['remaining_lease'] = (clean_df['remaining_lease'].str.split(' -',expand=True)[0].astype(float) * 12 + (clean_df['remaining_lease'].str.split(' -',expand=True).replace('',0)[1].astype(float))) # splits column, expands and sets values to float, multiplies years by 12 and adds the month value (also replaces blank spaces with 0 to catch values such as 5 years (no months)
print(clean_df['remaining_lease'].isnull().sum()) # making sure no values were missed 


print(clean_df)

0         11.0
1          2.0
2          2.0
3          5.0
4          2.0
          ... 
235803     2.0
235804     8.0
235805     8.0
235806     5.0
235807     8.0
Name: storey_range, Length: 235492, dtype: float64
0
        month        town         flat_type block        street_name  \
0           1  ANG MO KIO            2 ROOM   406  ANG MO KIO AVE 10   
1           1  ANG MO KIO            3 ROOM   108   ANG MO KIO AVE 4   
2           1  ANG MO KIO            3 ROOM   602   ANG MO KIO AVE 5   
3           1  ANG MO KIO            3 ROOM   465  ANG MO KIO AVE 10   
4           1  ANG MO KIO            3 ROOM   601   ANG MO KIO AVE 5   
...       ...         ...               ...   ...                ...   
235803      4      YISHUN         EXECUTIVE   827       YISHUN ST 81   
235804      5      YISHUN         EXECUTIVE   828       YISHUN ST 81   
235805      7      YISHUN         EXECUTIVE   877       YISHUN ST 81   
235806      5      YISHUN  MULTI-GENERATION   666       YISHUN

# 3.2 Checking Outliers

In [ ]:


print(clean_df[clean_df['resale_price'] == 1.7280000e+06]) # valid outlier (47th storey, Buikt Merah, with large remaining_lease)







        month         town flat_type block   street_name  storey_range  \
224726      4  BUKIT MERAH    5 ROOM   96A  HENDERSON RD          47.0   

        floor_area_sqm flat_model  lease_commence_date  remaining_lease  \
224726           113.0   Improved                 2019           1105.0   

        resale_price       date  year  
224726     1728000.0 2026-04-01  2026  
